In [1]:
from itertools import chain
from typing import Dict, List, NamedTuple

import numpy as np
from scipy.spatial.distance import cdist


Edge = NamedTuple('Edge', [('v', int), ('w', int)]) # edge contains two indices to coordinates-array
Cables = NamedTuple('Cables', [('costs', np.ndarray),
                               ('lengths', np.ndarray),
                               ('capacities', np.ndarray)])

In [2]:
CABLE_COSTS = np.array([0.58,0.87,1.24,1.95,3.13,5.19,6.9])
CABLE_LENGTHS = np.array([38.36,47.08,56.04,69.3,84.87,102.79,120.31])
CABLE_CAPACITIES = np.array([56,73,92,124,162,209,250])

In [3]:
cables = Cables(CABLE_COSTS, CABLE_LENGTHS, CABLE_CAPACITIES)

# intersection

In [4]:
def equal_coords(M, start_l, end_l, start_r, end_r):
    return np.all(M[:, start_l:end_l] == M[:,start_r:end_r], axis=1)


def disjoint_region(M, i, j, k, l):
    return np.all(np.maximum(M[:,i], M[:,j]) < np.minimum(M[:,k], M[:,l]))


def orientation(M, px, py, qx, qy, rx, ry) -> np.ndarray:
    val = (M[:,qy]-M[:,py]) * (M[:,rx]-M[:,qx]) - (M[:,qx]-M[:,px]) * (M[:,ry]-M[:,qy])
    np.place(val, val > 0, 1)
    np.place(val, val < 0, 2)
    return val


def on_segment(M, px, py, qx, qy, rx, ry) -> np.ndarray:
    return np.logical_and.reduce((M[:,qx] <= np.maximum(M[:,px], M[:,rx]),
                                  M[:,qx] >= np.minimum(M[:,px], M[:,rx]),
                                  M[:,qy] <= np.maximum(M[:,py], M[:,ry]),
                                  M[:,qy] >= np.minimum(M[:,py], M[:,ry])))


def is_edge_intersecting(edge: np.ndarray, edge_set: List[np.ndarray], partition_indices: List[bool] = []) -> bool:
    """
    :param edge: array of size (1,4)
    :param edge_set: array of shape (N,4)
    :param partition_indices: boolean list that selects subset of edge_set
    """
    if not partition_indices:
        edge_set = np.vstack(edge_set)
    else:
        edge_set = np.vstack(compress(edge_set, partition_indices))

    # NOTE: ugly hack
    if len(edge_set) == 0:
        return False

    M = np.concatenate((np.repeat(edge, len(edge_set), axis=0), edge_set), axis=1)

    # obtain masking of incident edges
    incident_mask = np.logical_or.reduce((equal_coords(M,0,2,4,6),
                                          equal_coords(M,0,2,6,8),
                                          equal_coords(M,2,4,4,6),
                                          equal_coords(M,2,4,6,8)))
    M = M[np.logical_not(incident_mask)]
    if M.size == 0:
        return False

    o1 = orientation(M,0,1,2,3,4,5)
    o2 = orientation(M,0,1,2,3,6,7)
    o3 = orientation(M,4,5,6,7,0,1)
    o4 = orientation(M,4,5,6,7,2,3)

    if np.logical_and(o1 != o2, o3 != o4).any():
        return True

    if np.logical_and(o1 == 0, on_segment(M,0,1,4,5,2,3)).any():
        return True
    if np.logical_and(o2 == 0, on_segment(M,0,1,6,7,2,3)).any():
        return True
    if np.logical_and(o3 == 0, on_segment(M,4,5,0,1,6,7)).any():
        return True
    if np.logical_and(o4 == 0, on_segment(M,4,5,2,3,6,7)).any():
        return True
    return False


# assign power cables

In [21]:
from itertools import chain
from typing import Dict, List, Set

import numpy as np
from scipy.spatial.distance import cdist


def get_successors(edges: List[List[Edge]], N) -> Dict[int,List[int]]:
    successor = {i:[] for i in range(N)}
    for e in chain.from_iterable(edges):
        successor[e.v].append(e.w)
    return successor

def get_successors_gen(v, successors):
    res = [v]
    for s in successors[v]:
        res = res + get_successors_gen(s, successors)
    return res
        
def get_successor_list(edges: List[List[Edge]], N):
    succs = get_successors(edges,N)
    res = {i:[] for i in range(N)}
    for i in range(N):
        res[i] = get_successors_gen(i,succs)
    return res

def get_predecessor(edges: List[List[Edge]]) -> Dict[int,int]:
    predecessor = {}
    for e in chain.from_iterable(edges):
        predecessor[e.w] = e.v
    return predecessor

def obtain_parents(edges: List[List[Edge]], N) -> Dict[int,List[int]]:
    predecessor = get_predecessor(edges)
    parents = {i:[] for i in range(N)} # type: Dict[int,List[int]]
    for i in range(1,N):
        parents[i] = list(get_predecessors(i,predecessor))
    return parents
    
def get_predecessors(v, predecessor):
    while(predecessor[v] != 0):
        yield predecessor[v]
        v = predecessor[v]


def obtain_leaves(edges: List[List[Edge]]) -> List[int]:
    successor = get_successors(edges)
    return [v for v in successor if successor[v] == []]


def count_successors(vertex, successors) -> int:
    i = 1
    for s in successors[vertex]:
        i+= count_successors(s, successors)
    return i


def compute_capacities(edges, N) -> np.ndarray:
    successor = get_successors(edges,N)
    return np.array([count_successors(i,successor) for i in range(N)])


def assign_power_cables(edges: List[List[Edge]], coordinates, cable_lengths, cable_capacities) -> Dict[Edge,int]:
    capacities = compute_capacities(edges, coordinates.shape[0])
    distances = cdist(coordinates, coordinates)

    assignment = {}

    for e in chain.from_iterable(edges):
        length = distances[e.v,e.w] <= cable_lengths
        capacity = capacities[e.w] <= cable_capacities

        best_cable_index = np.where(np.logical_and(length,capacity) == True)
        if best_cable_index[0].size == 0:
            raise RuntimeError('No suitable cable found for {}'.format(e))

        assignment[e] = best_cable_index[0][0]
    return assignment

def assign_cable(power_cable_assignment,N):
    cables = np.zeros(N)
    for edge, cable_index in power_cable_assignment.items():
        cables[edge.w] = cable_index
    cables[0] = np.nan
    return cables.astype(int)

def compute_max_capacities(power_cable_assignment,N) -> np.ndarray:
    max_capacities = np.zeros(N)
    for edge, cable_index in power_cable_assignment.items():
        max_capacities[edge.w] = CABLE_CAPACITIES[cable_index]
    max_capacities[0] = N
    return max_capacities.astype(int)

# solution class

In [6]:
def _switch_cost(degree: int) -> float:
        """Note: Hard-coded values for quick and dirty development"""
        if degree <= 2:
            return 100
        elif degree >= 3 and degree <= 9:
            return 800
        elif degree >= 10 and degree <= 17:
            return 1500
        else:
            raise ValueError('Degree is too high. No cost found.')

In [7]:
def total_cost(edges, degrees, data_cable_cost, power_cable_assignment):
    data_cost = sum(D[e.v,e.w]*data_cable_cost for e in chain.from_iterable(edges)) #incl trench cost + foil
    switch_cost = sum(_switch_cost(d) for d in degrees.values()) - _switch_cost(degrees[0])
    power_cost = sum(2*D[e.v,e.w] * CABLE_COSTS[power_cable_assignment[e.w]] for e in chain.from_iterable(edges))
    return data_cost + switch_cost + power_cost

# Local Search

In [30]:
from itertools import chain, product

class LocalSearchUpgradeDowngrade:
    
    def __init__(self, edges: List[List[Edge]],
                 edges_coordinates: np.ndarray,
                 degrees: Dict[int,int],
                 cables: Cables,
                 data_cable_cost: float,
                 coordinates: np.ndarray):
        self.edges = edges
        self.edge_set = set(chain.from_iterable(self.edges))
        self.edge_partition = {}
        for i in range(len(self.edges)):
            for e in self.edges[i]:
                self.edge_partition[e] = i
        
        self.edges_coordinates = edges_coordinates
        self.degrees = degrees
        
        self.power_cable_assignment = assign_power_cables(edges, coordinates, cables.lengths, cables.capacities)
        self.capacities = compute_capacities(edges, coordinates.shape[0]) # type: np.ndarray
        self.max_capacities = compute_max_capacities(self.power_cable_assignment, coordinates.shape[0])
        self.cable_assignment = assign_cable(self.power_cable_assignment, coordinates.shape[0])
        
        self.parents = obtain_parents(edges, coordinates.shape[0])
        self.predecessor = get_predecessor(edges)
        self.all_successors = get_successor_list(edges, coordinates.shape[0]) 
        for i in range(coordinates.shape[0]):
            self.all_successors[i].remove(i) # remove oneself
        
        self.data_cable_cost = data_cable_cost
        self.cables = cables # assume all are ordered ascendingly        
        self.coordinates = coordinates
        
        self.distance = cdist(coordinates, coordinates)
        self.heliostats = range(1,coordinates.shape[0])
        
    def perform_local_search(self):
        i = 0
        improv = 0
        while True:
            candidate = self._next_candidate()
            if candidate is None:
                break
            improv += candidate[0]
            self._update(candidate)

            plot_solution(coordinates, '/home/duc/mst_{}.png'.format(i), self.edges, 0, 0, 3)
            i += 1
        print("Total improvement: {}".format(improv))
            
    def _update(self,candidate):
        total, edge, new_edge_cable, parents_upgrade, cable_indices_upgrade, parents_downgrade, cable_indices_downgrade = candidate
        
        print('Improve by {}'.format(total))
        
        # remove old edge
        old_edge = Edge(self.predecessor[edge.w], edge.w)
        
        p = self.edge_partition[old_edge]
        self.edges[p].remove(old_edge)
        self.degrees[old_edge.v] -= 1
        del self.edge_partition[old_edge]
        self.edge_set.remove(old_edge)
        
        index = np.where(np.all(self._coordinates(old_edge) == self.edges_coordinates[p], axis=1))
        self.edges_coordinates[p] = np.delete(self.edges_coordinates[p], index, axis=0)
        self.predecessor[edge.w] = edge.v # new predecessor relation
        
        
        for q in self.parents[edge.w]:
            self.all_successors[q] = list(set(self.all_successors[q]) - set(self.all_successors[edge.w]))
            self.all_successors[q].remove(edge.w)
        
        # add edge
        for s in self.all_successors[edge.w]:
            for old_parent in self.parents[edge.w]:
                self.parents[s].remove(old_parent)
            self.parents[s] = self.parents[s] + self.parents[edge.v] + [edge.v]
            
        self.parents[edge.w] = self.parents[edge.v] + [edge.v]
        
        for parent in self.parents[edge.v]:
            self.all_successors[parent] = self.all_successors[parent] + self.all_successors[edge.w] + [edge.w]
        
        self.all_successors[edge.v] += self.all_successors[edge.w] + [edge.w]
            
        partition = self.edge_partition[self.predecessor[edge.v], edge.v]
        self.edges[partition].append(edge)
        self.edge_set.add(edge)
        
        self.cable_assignment[edge.w] = new_edge_cable
        self.capacities[edge.v] += self.capacities[edge.w]
        self.degrees[edge.v] += 1
        self.edge_partition[edge] = partition
        
        self.edges_coordinates[partition] = np.vstack((self.edges_coordinates[partition], self._coordinates(edge)))

        # upgrade parents of edge.v
        for parent,new_parent_cable in zip(parents_upgrade, cable_indices_upgrade):
            self.cable_assignment[parent] = new_parent_cable
            self.capacities[parent] += self.capacities[edge.w]
            self.max_capacities[parent] = self.cables.capacities[new_parent_cable]
        
        # downgrade parents of edge.w
        for parent,new_parent_cable in zip(parents_downgrade, cable_indices_downgrade):
            self.cable_assignment[parent] = new_parent_cable
            self.capacities[parent] -= self.capacities[edge.w]
            self.max_capacities[parent] = self.cables.capacities[new_parent_cable]
        
        for v in self.heliostats:
            assert len(self.parents[v]) < self.coordinates.shape[0]
            assert len(self.all_successors[v]) < self.coordinates.shape[0]
            
        
    def _next_candidate(self):
        for v,w in product(self.heliostats, self.heliostats):
            if v != w and self._valid_edge(v,w):
                # upgrade costs
                upgrade_cost, parents_upgrade, cable_indices_upgrade = self._upgrade_power_cables(v,w)
                switch_upgrade = self._upgrade_switch(v)
                
                # downgrade costs
                downgrade_profit, parents_downgrade, cable_indices_downgrade = self._downgrade_power_cables(w)
                
                # new edge
                cost_new_edge, new_edge_cable = self._new_edge(v,w)
                
                # deleted edge
                profit_deleted_edge = self._delete_edge(w)
                
                total = upgrade_cost + switch_upgrade + cost_new_edge - downgrade_profit - profit_deleted_edge
                if total < 0: # first improvement
                    candidate = (total,
                                  Edge(v,w),
                                  new_edge_cable,
                                  parents_upgrade,
                                  cable_indices_upgrade,
                                  parents_downgrade,
                                  cable_indices_downgrade)
                    return candidate
        return None
                             
    def _valid_edge(self, v, w):
        return self.distance[v,w] <= self.cables.lengths[-1] and \
               not is_edge_intersecting(self._coordinates(Edge(v,w)), self.edges_coordinates) and \
               Edge(v,w) not in self.edge_set and \
               self.predecessor[w] != 0 and \
                Edge(w,v) not in self.edge_set and \
                w not in self.parents[v]
    
    ### NEW EDGE #########################
    def _new_edge(self, v, w):
        # get cable for v-> w s.t. capacity of w can be covered + length restriction
        cap = self.capacities[w] < self.cables.capacities
        length = self.distance[v,w] < self.cables.lengths
        
        res = np.where(np.logical_and(cap,length) == True)[0]
        if res.size == 0:
            return RuntimeError('New Edge error!')
        
        power_cable_cost = self.cables.costs[res[0]] * self.distance[v,w] * 2
        data_dable_cost = self.distance[v,w] * self.data_cable_cost
        return power_cable_cost+data_dable_cost, res[0]
    
    ### DELETED EDGE #########################
    def _delete_edge(self, w):
        power_cable_cost = self.distance[self.predecessor[w],w] * self.cables.costs[self.cable_assignment[w]] * 2
        data_cable_cost = self.distance[self.predecessor[w],w] * self.data_cable_cost
        if self.degrees[self.predecessor[w]] == 3 or self.degrees[self.predecessor[w]] == 10:
            data_cable_cost += 700
            
        return power_cable_cost + data_cable_cost
    
    ### UPGRADE COST #########################
    def _upgrade_switch(self, v):
        if self.degrees[v] == 2 or self.degrees[v] == 9:
            return 700
        return 0
    
    def _upgrade_power_cables(self, v, w):
        cost = 0
        cables_upgrade = []
        for parent in self.parents[v]:
            new_cable_index = self._upgrade_cable_index(parent, w)
            if new_cable_index == -1:
                return np.inf, None, None
            cables_upgrade.append(new_cable_index)
            cost += self._cable_cost_difference_upgrade(parent, new_cable_index)
        return cost, self.parents[v], cables_upgrade
                    
    def _upgrade_cable_index(self, parent, w):
        new_cap = self.capacities[parent] + self.capacities[w]
        if new_cap > self.cables.capacities[-1]:
            return -1
        
        valid_cables = new_cap <= self.cables.capacities
        valid_lengths = self.distance[self.predecessor[parent],parent] <= self.cables.lengths

        res = np.where(np.logical_and(valid_cables, valid_lengths) == True)[0]
        if res.size == 0:
            raise RuntimeError('SHOULD NOT BE REACHABLE ANYMORE. No suitable cable found for {}. Additional capacity:'.format(parent,self.capacities[w]))
        return res[0]
            
    def _cable_cost_difference_upgrade(self, parent, new_cable):
        cost_difference = self.cables.costs[new_cable] - self.cables.costs[self.cable_assignment[parent]]
        return self.distance[self.predecessor[parent], parent] * cost_difference * 2
    #########################
    
    ### DOWNGRADE PROFIT #########################
    def _downgrade_power_cables(self, w):
        profit = 0
        cables_downgrade = []
        for parent in self.parents[w]:
            new_cable_index = self._downgrade_cable_index(parent, w)
            if new_cable_index == -1:
                return -np.inf, None, None
            cables_downgrade.append(new_cable_index)
            profit += self._cable_cost_difference_downgrade(parent, new_cable_index)
        return profit, self.parents[w], cables_downgrade
    
    def _downgrade_cable_index(self, parent, w):
        new_cap = self.capacities[parent] - self.capacities[w]
        valid_cables = new_cap <= self.cables.capacities
        valid_lengths = self.distance[self.predecessor[parent],parent] <= self.cables.lengths

        res = np.where(np.logical_and(valid_cables, valid_lengths) == True)[0]
        if res.size == 0:
            return -1
            #raise RuntimeError('No suitable cable found')
        return res[0]
    
    def _cable_cost_difference_downgrade(self, parent, new_cable):
        cost_difference = self.cables.costs[self.cable_assignment[parent]] - self.cables.costs[new_cable]
        return self.distance[self.predecessor[parent], parent] * cost_difference * 2
    #########################
        
    def _coordinates(self, edge: Edge):
        return np.array([[self.coordinates[edge.v][0],
                          self.coordinates[edge.v][1],
                          self.coordinates[edge.w][0],
                          self.coordinates[edge.w][1]]])

## TEST HAMILTON

In [9]:
class EdgeSolution:
    def __init__(self, partitions: int) -> None:
        """
        :param partitions: number of edge partitions
        """
        self.edges= [[] for _ in range(partitions)] # type: List[List[Edge]]
        self.edges_coords = [np.empty((0,4)) for _ in range(partitions)]
        self.edge_costs = {} # type: Dict[Edge,int]

    def add_edge(self, edge: Edge, edge_cost: float, partition: int, coordinates: np.ndarray):
        self.edges[partition].append(edge)
        self.edge_costs[edge] = edge_cost

        edges_coords = np.array([[coordinates[edge.v][0], coordinates[edge.v][1],
                                 coordinates[edge.w][0], coordinates[edge.w][1]]])
        self.edges_coords[partition] = np.vstack((self.edges_coords[partition], edges_coords))

    def remove_edge(self, edge: Edge, partition: int, coordinates: np.ndarray):
        self.edges[partition].remove(edge)
        del self.edge_costs[edge]

        edges_coords = np.array([[coordinates[edge.v][0], coordinates[edge.v][1],
                                 coordinates[edge.w][0], coordinates[edge.w][1]]])
        delete_index = np.where(np.all(edges_coords == self.edges_coords[partition], axis=1))
        self.edges_coords[partition] = np.delete(self.edges_coords[partition], delete_index, axis=0)

    def intersects(self, edges_coords: np.ndarray) -> bool:
        """
        Checks whether a given edge that is represented by its coordinates is intersecting
        with our solution edges.

        :param edges_coords: a numpy array that holds both x- and y-coordinates
        :returns: True if edges_coords intersects with solution edges
        """
        return is_edge_intersecting(edges_coords, self.edges_coords)

    def cost(self, distances: np.ndarray):
        return sum(distances[e.v,e.w]*self.edge_costs[e] for e in chain.from_iterable(self.edges))


class DataCableSolution(EdgeSolution):
    def __init__(self, n: int, partitions: int):
        super().__init__(partitions)
        self.degrees = {i:0 for i in range(n)} # type: Dict[int,int]

    def add_edge(self, edge: Edge, edge_cost: float, partition: int, coordinates: np.ndarray):
        super().add_edge(edge, edge_cost, partition, coordinates)
        self.degrees[edge.v] += 1
        self.degrees[edge.w] += 1

    def remove_edge(self, edge: Edge, partition: int, coordinates: np.ndarray):
        super().remove_edge(edge, partition, coordinates)
        self.degrees[edge.v] -= 1
        self.degrees[edge.w] -= 1

    def cost(self, distances: np.ndarray):
        cost = super().cost(distances)
        # subtract switch costs of solar tower at the end
        # NOTE: make it more readable or omit switch cost of solar tower immediately
        return cost + sum(self._switch_cost(d) for d in self.degrees.values()) - self._switch_cost(self.degrees[0])

    def _switch_cost(self, degree: int) -> float:
        """Note: Hard-coded values for quick and dirty development"""
        if degree <= 2:
            return 100
        elif degree >= 3 and degree <= 9:
            return 800
        elif degree >= 10 and degree <= 17:
            return 1500
        else:
            raise ValueError('Degree is too high. No cost found.')


In [10]:
#!/usr/bin/env python3
# UTF-8 encoding
from typing import Dict, List, Set, NamedTuple

import numpy as np
from recordclass import recordclass
from scipy.spatial.distance import cdist


# TODO: document these NamedTuples
# Candidate = RecordClass('Candidate', [('cost', float),
#                                       ('edges', List[Edge]),
#                                       ('vertex', int),
#                                       ('index', int)])
# HamiltonState = RecordClass('HamiltonState', [('permutation', List[int]),
#                                               ('unvisited', Set[int])])

# TODO: buggy pip install - RecordClass not in package?? quick hack
Candidate = recordclass('Candidate', ['cost',
                                      'edges',
                                      'vertex',
                                      'index'])
HamiltonState = recordclass('HamiltonState', ['permutation',
                                              'unvisited'])


def shift_left(L: List[int]):
    """
    Works faster than np.roll(L, -1)
    """
    return L[1:] + L[:1]


class Hamilton:
    """
    We compute Hamiltonian paths for each partition that are initial solutions for the
    data cabling.
    """
    __slots__ = ('coordinates', 'cable_cost', 'edge_costs', 'partitions', 'solution', 'current_state')

    def __init__(self, coordinates: np.ndarray, cable_cost: float, partitions: int) -> None:
        """
        :param coordinates: coordinates vertices
        :param cable: cost for hamiltonian path
        :param partitions: number of partitions
        """
        self.coordinates = coordinates
        self.cable_cost = cable_cost
        self.edge_costs = cdist(self.coordinates, self.coordinates) * cable_cost # only consider glass fiber cables!
        self.partitions = partitions
        self.solution = DataCableSolution(self.coordinates.shape[0], partitions)
        self.current_state = HamiltonState(None, None)

    def compute(self) -> None:
        """
        Compute Hamiltonian paths for each partition.
        """
        for i, p in enumerate(compute_partitions(coordinates=self.coordinates, partitions=self.partitions)):
            self._compute_hamilton(i, p)

    def _compute_hamilton(self, partition: int, partition_indices: np.ndarray) -> None:
        """
        Computes for a partition of indices a valid solution regarding planarity.

        :param partition_indices: indices of heliostats that belong together in a partition
        :return: edges and edge_coords that form a Hamiltonian path
        """
        self.current_state._replace(permutation=[0])
        self.current_state._replace(unvisited=set(partition_indices.astype(int).flat))

        while len(self.solution.edges[partition]) < partition_indices.shape[0]:
            best_candidate = min(self._compute_candidates())
            if len(best_candidate.edges) == 2:
                self._insert_between(best_candidate, partition)
            else:
                self._insert_last(best_candidate, partition)

    def _compute_candidates(self) -> Candidate:
        """
        Compute cost and edges for a vertex and index position of permutation. In other words
        for a vertex v we compute its minimum cost of insertion into the current permutation
        of our Hamiltonian path. This can either be an insertion and appending to the permutation.

        :param vertex: vertex to be added to permutation
        :param index: index of insertion/appending
        """
        for vertex in self.current_state.unvisited:
            shifted_perm = shift_left(self.current_state.permutation)
            unvisited = [vertex]*len(self.current_state.permutation)

            costs_edge_two = self.edge_costs[ unvisited, shifted_perm]
            costs_edge_two[-1] = 0
            adding_costs = self.edge_costs[ unvisited, self.current_state.permutation] + costs_edge_two

            cost_permutation_edges = self.edge_costs[self.current_state.permutation, shifted_perm]
            cost_permutation_edges[-1] = 0
            index = np.argmin(adding_costs - cost_permutation_edges)

            if index < len(self.current_state.permutation)-1:
                e1 = Edge(self.current_state.permutation[index], vertex)
                e2 = Edge(vertex, self.current_state.permutation[index+1])
                d = Edge(self.current_state.permutation[index], self.current_state.permutation[index+1])

                cost = self.edge_costs[e1.v, e1.w] + self.edge_costs[e2.v, e2.w] - self.edge_costs[d.v, d.w]
                yield Candidate(cost=cost, edges=[e1,e2], vertex=vertex, index=index)
            else:
                e = Edge(self.current_state.permutation[index], vertex)
                yield Candidate(cost=self.edge_costs[e.v, e.w], edges=[e], vertex=vertex, index=index)

    def _insert_edge(self, edge: Edge, partition: int) -> None:
        """
        Adds edge to solution partition
        """
        self.solution.add_edge(edge=edge,
                               edge_cost=self.cable_cost,
                               partition=partition,
                               coordinates=self.coordinates)
        # update current unvisited heliostats set
        for vertex in edge:
            if vertex in self.current_state.unvisited:
                self.current_state.unvisited.remove(vertex)

    def _insert_between(self, candidate: Candidate, partition: int) -> None:
        # remove old edge
        index = candidate.index
        old_edge = Edge(self.current_state.permutation[index], self.current_state.permutation[index+1])
        self.solution.remove_edge(old_edge, partition, self.coordinates)
        # add new edges
        self._insert_edge(candidate.edges[0], partition)
        self._insert_edge(candidate.edges[1], partition)
        # update current permutation
        self.current_state.permutation.insert(candidate.index+1, candidate.vertex)

    def _insert_last(self, candidate: Candidate, partition: int) -> None:
        self._insert_edge(candidate.edges[0], partition)
        self.current_state.permutation.append(candidate.vertex)


In [11]:
def compute_partitions(coordinates: np.ndarray, partitions: int) -> List[np.ndarray]:
    """
    Partitions the coordinates into an array of indices. For this matter we use the angles
    between the reference vector of the x-axis.

    :param coordinates: a 2D real array of x- and y-coordinates
    :param partitions: number of partitions
    :return: list of arrays that contain indices of heliostats with regards to the coordinates array
    """
    if partitions > len(coordinates) - 1: # not enough heliostats
        return []

    degrees = np.degrees(np.arctan2(coordinates[1:,1], coordinates[1:,0]))
    indices = degrees.argsort()
    padding = (-len(indices))%partitions
    L = np.split(np.concatenate((indices,np.ones(padding)*-1)),partitions) # padding value: -1
    # remove padding from last partition and increase index
    return [np.delete(a, np.where(a == -1)) + 1 for a in L]

In [12]:
coordinates = np.loadtxt('../data/instances/PS10.csv', delimiter=';')

In [13]:
coordinates = np.vstack((np.array([0,0]), coordinates))

In [14]:
ham = Hamilton(coordinates, 29, 3)

In [1588]:
ham.compute()

In [1589]:
ls = LocalSearchUpgradeDowngrade(ham.solution.edges, ham.solution.edges_coords, ham.solution.degrees, cables, 54, coordinates)

In [1596]:
total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)

583646.9927672277

In [1597]:
ls.perform_local_search()

Total improvement: 0


## MST

In [15]:
import sys
from itertools import product
from typing import List, Tuple, Set

import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

class MSTPrim:
    def __init__(self, coordinates: np.ndarray, cable_cost: int) -> None:
        self.coordinates = coordinates
        self.cable_cost = cable_cost
        self.edge_costs = cdist(self.coordinates, self.coordinates) * self.cable_cost + 100 # cable and conductor costs
        self.degrees = {i:0 for i in range(self.coordinates.shape[0])}
        self.successor = {i:[] for i in range(self.coordinates.shape[0])}
        self.predecessor = {i:[] for i in range(self.coordinates.shape[0])}

        self.edges = [] # type: List[List[Edge]]
        self.edge_coords = []

    def compute(self, partitions: int) -> None:
        self.partitions = compute_partitions(coordinates=self.coordinates, partitions=partitions)
        for p in self.partitions:
            self.edges.append(self._compute_mst(p))

    def _compute_mst(self, partition_indices: np.ndarray) -> List[Edge]:
        """
        Computes for a partition of indices a valid solution regarding planarity.
        TODO: extensive documentation
        """
        # TODO: introduce Current namedTuple for these self-variables
        # initialization
        self.visited_heliostats = set([0])
        self.unvisited_heliostats = {int(i) for i in partition_indices}

        partition_edges = [] # forming a MST
        partition_edges_coords = np.empty((0,4))

        while len(partition_edges) < partition_indices.shape[0]:
            # select min cost edge
            edges = self._next_cut_edges()
            if not edges:
                raise RuntimeError('empty edge candidates list. WEIRD ERROR')

            # TODO: looks ugly - refactoring needed
            updated = False
            for e in edges:
                e_coords = np.array([[self.coordinates[e.v][0],
                                     self.coordinates[e.v][1],
                                     self.coordinates[e.w][0],
                                     self.coordinates[e.w][1]]])
                if not is_edge_intersecting(e_coords, [partition_edges_coords]):
                    partition_edges.append(e)
                    partition_edges_coords = np.vstack((partition_edges_coords, e_coords))
                    self.successor[e.v].append(e.w)
                    self.predecessor[e.w].append(e.v)

                    self._update(e)
                    updated = True
                    break
            if not updated:
                raise RuntimeError('SHOULD NOT HAPPEN')
        self.edge_coords.append(partition_edges_coords)
        return partition_edges


    def _next_cut_edges(self) -> List[Edge]:
        edges = [Edge(*t) for t in product(self.visited_heliostats, self.unvisited_heliostats)]
        costs = [self.edge_costs[e.v][e.w] for e in edges]
        return [x for _,x in sorted(zip(costs,edges))]

    def _update(self, edge: Edge) -> None:
        for vertex in edge:
            self.degrees[vertex] += 1

            for h in self.unvisited_heliostats:
                if self.degrees[vertex] == 2 or self.degrees[vertex] == 9:
                    self.edge_costs[vertex][h] += 700
                    self.edge_costs[h][vertex] += 700

                if self.degrees[vertex] == 3 or self.degrees[vertex] == 10:
                    self.edge_costs[vertex][h] -= 700
                    self.edge_costs[h][vertex] -= 700

            self.visited_heliostats.add(vertex)
            if vertex in self.unvisited_heliostats:
                self.unvisited_heliostats.remove(vertex)


In [18]:
mst = MSTPrim(coordinates[:heliostat_count], 54)
mst.compute(3)
plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)

NameError: name 'plot_solution' is not defined

In [ ]:
def solution_value(distances: np.ndarray, edges, edge_cost: float, degrees):
    cost = sum(distances[e.v,e.w]*edge_cost for e in chain.from_iterable(edges))
    # subtract switch costs of solar tower at the end
    # NOTE: make it more readable or omit switch cost of solar tower immediately
    return cost + sum(_switch_cost(d) for d in degrees.values()) - _switch_cost(degrees[0])

In [ ]:
solution_value(cdist(coordinates,coordinates), mst.edges, 54, mst.degrees)

In [17]:
heliostat_count = 625

In [1627]:
mst = MSTPrim(coordinates[:heliostat_count], 29)
mst.compute(9)
plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 29, coordinates[:heliostat_count])
original_cost = total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)
print("TOTAL BEFORE {}".format(original_cost))

ls.perform_local_search()

new_cost = total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)
print("TOTAL AFTER {}".format(new_cost))
print('IMPROV: {}'.format(original_cost-new_cost))
print(Counter(ls.degrees.values()))
print()
    

TOTAL BEFORE 619591.84959812
Improve by -116.95743148123074
Total improvement: -116.95743148123074
TOTAL AFTER 619291.6223322767
IMPROV: 300.22726584330667
Counter({2: 535, 1: 52, 3: 32, 4: 4, 9: 1, 5: 1})



In [1624]:
solution_value(cdist(coordinates,coordinates), mst.edges, 14, mst.degrees)

331196.94745101983

In [25]:
D = cdist(coordinates,coordinates)

In [27]:
from collections import Counter

In [35]:
mst = MSTPrim(coordinates[:heliostat_count], 29)
mst.compute(3)
plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 29, coordinates[:heliostat_count])
original_cost = total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)
print("TOTAL BEFORE {}".format(original_cost))

ls.perform_local_search()

new_cost = total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)
print("TOTAL AFTER {}".format(new_cost))
print('IMPROV: {}'.format(original_cost-new_cost))
print(Counter(ls.degrees.values()))
print()


TOTAL BEFORE 620277.5299264379
Improve by -1029.2168966203024
Improve by -528.1056425369688
Improve by -1029.2168966203024
Improve by -622.4959325549526
Improve by -654.4064526221083
Improve by -654.4064526221083
Improve by -250.35544390012933
Improve by -0.14771233073599888
Improve by -7.290579014206628
Improve by -250.35544390012933
Improve by -0.14771233073599888
Improve by -7.290579014206628
Improve by -125.2272086470191
Improve by -76.17635881178239
Improve by -148.09914228386288
Improve by -81.61387388490562
Improve by -391.2168385074144
Improve by -218.96163920404638
Total improvement: -6074.730805405918
TOTAL AFTER 611592.3934861188
IMPROV: 8685.136440319126
Counter({2: 565, 1: 37, 3: 12, 4: 10, 5: 1})



In [34]:
mst = MSTPrim(coordinates[:heliostat_count], 14)
mst.compute(3)
plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 14, coordinates[:heliostat_count])
original_cost = total_cost(ls.edges, ls.degrees, 14, ls.cable_assignment)
print("TOTAL BEFORE {}".format(original_cost))

ls.perform_local_search()

new_cost = total_cost(ls.edges, ls.degrees, 14, ls.cable_assignment)
print("TOTAL AFTER {}".format(new_cost))
print('IMPROV: {}'.format(original_cost-new_cost))
print(Counter(ls.degrees.values()))
print()


TOTAL BEFORE 376851.9646464674
Improve by -1017.8384290116533
Improve by -600.0812372956368
Improve by -97.39213039014544
Improve by -312.1612669622484
Improve by -874.4745295711167
Improve by -1017.8384290116533
Improve by -688.5001620701919
Improve by -540.9606483542939
Improve by -1017.8384290116533
Improve by -12.820063143377865
Improve by -30.775366596286403
Improve by -630.6605900288882
Improve by -180.47702600300943
Improve by -111.75400722953441
Improve by -795.4895113127228
Improve by -661.0423851967201
Improve by -176.76432183505824
Improve by -25.752038319280928
Improve by -23.15659834686835
Improve by -220.9466700900383
Improve by -47.90337743924931
Improve by -82.4014831866516
Improve by -107.23084557934635
Improve by -1529.821424772075
Improve by -39.49906568023812
Improve by -8.756091496816282
Improve by -75.61023612480295
Improve by -1.835396737193605
Improve by -16.59107526992443
Improve by -112.26067062787763
Total improvement: -11058.633506694552
TOTAL AFTER 362567.3

In [33]:

mst = MSTPrim(coordinates[:heliostat_count], 54)
mst.compute(3)
plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 54, coordinates[:heliostat_count])
original_cost = total_cost(ls.edges, ls.degrees, 54, ls.cable_assignment)
print("TOTAL BEFORE {}".format(original_cost))

ls.perform_local_search()

new_cost = total_cost(ls.edges, ls.degrees, 54, ls.cable_assignment)
print("TOTAL AFTER {}".format(new_cost))
print('IMPROV: {}'.format(original_cost-new_cost))
print(Counter(ls.degrees.values()))
print()
    

TOTAL BEFORE 1032219.393886046
Improve by -1048.1810093013842
Improve by -1048.1810093013842
Improve by -99.88469141737687
Improve by -305.52449756890405
Improve by -327.59819611555054
Improve by -109.78077383128561
Improve by -601.0491765242233
Improve by -154.57019816890784
Improve by -228.62488577712452
Improve by -225.75088011940397
Improve by -225.75088011940397
Improve by -89.26395651773396
Improve by -268.12877618859443
Improve by -217.3164607827191
Improve by -33.33617544201161
Improve by -146.1927288466486
Improve by -21.98313044763654
Improve by -149.2646314154972
Improve by -421.5731529848797
Total improvement: -5721.955210870671
TOTAL AFTER 1021309.8827088577
IMPROV: 10909.511177188368
Counter({2: 551, 1: 42, 3: 24, 4: 8})



In [28]:
for i in range(3,10):
    mst = MSTPrim(coordinates[:heliostat_count], 54)
    mst.compute(i)
    plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
    ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 54, coordinates[:heliostat_count])
    original_cost = total_cost(ls.edges, ls.degrees, 54, ls.cable_assignment)
    print("TOTAL BEFORE {}".format(original_cost))
    
    ls.perform_local_search()
    
    new_cost = total_cost(ls.edges, ls.degrees, 54, ls.cable_assignment)
    print("TOTAL AFTER {}".format(new_cost))
    print('IMPROV: {}'.format(original_cost-new_cost))
    print(Counter(ls.degrees.values()))
    print()
    

TOTAL BEFORE 1032219.393886046
Improve by -1048.1810093013842
Improve by -1048.1810093013842
Improve by -99.88469141737687
Improve by -305.52449756890405
Improve by -327.59819611555054
Improve by -109.78077383128561
Improve by -601.0491765242233
Improve by -154.57019816890784
Improve by -228.62488577712452
Improve by -225.75088011940397
Improve by -225.75088011940397
Improve by -89.26395651773396
Improve by -268.12877618859443
Improve by -217.3164607827191
Improve by -33.33617544201161
Improve by -146.1927288466486
Improve by -21.98313044763654
Improve by -149.2646314154972
Improve by -421.5731529848797
Total improvement: -5721.955210870671
TOTAL AFTER 1021309.8827088577
IMPROV: 10909.511177188368
Counter({2: 551, 1: 42, 3: 24, 4: 8})

TOTAL BEFORE 1031032.6274923367
Improve by -232.08804802902182
Improve by -242.41519496560818
Improve by -232.08804802902182
Improve by -558.2518448186731
Improve by -36.71149543290289
Improve by -36.71149543290289
Improve by -214.78596351843453
Improve 

SystemError: <built-in method write of _io.BufferedWriter object at 0x7f9c4e3a3518> returned a result with an error set

In [ ]:
for i in range(3,10):
    mst = MSTPrim(coordinates[:heliostat_count], 29)
    mst.compute(i)
    plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
    ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 29, coordinates[:heliostat_count])
    original_cost = total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)
    print("TOTAL BEFORE {}".format(original_cost))
    
    ls.perform_local_search()
    
    new_cost = total_cost(ls.edges, ls.degrees, 29, ls.cable_assignment)
    print("TOTAL AFTER {}".format(new_cost))
    print('IMPROV: {}'.format(original_cost-new_cost))
    print(Counter(ls.degrees.values()))
    print()
    

In [ ]:
for i in range(3,10):
    mst = MSTPrim(coordinates[:heliostat_count], 14)
    mst.compute(i)
    plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
    ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 14, coordinates[:heliostat_count])
    original_cost = total_cost(ls.edges, ls.degrees, 14, ls.cable_assignment)
    print("TOTAL BEFORE {}".format(original_cost))
    
    ls.perform_local_search()
    
    new_cost = total_cost(ls.edges, ls.degrees, 14, ls.cable_assignment)
    print("TOTAL AFTER {}".format(new_cost))
    print('IMPROV: {}'.format(original_cost-new_cost))
    print(Counter(ls.degrees.values()))
    print()
    

In [1613]:
for i in range(3,10):
    mst = MSTPrim(coordinates[:heliostat_count], 14)
    mst.compute(i)
    plot_solution(coordinates, '/home/duc/mst.png', mst.edges, 0, 0, 3)
    ls = LocalSearchUpgradeDowngrade(mst.edges, mst.edge_coords, mst.degrees, cables, 14, coordinates[:heliostat_count])
    original_cost = total_cost(ls.edges, ls.degrees, 14, ls.cable_assignment)
    print("TOTAL BEFORE {}".format(original_cost))
    
    ls.perform_local_search()
    
    new_cost = total_cost(ls.edges, ls.degrees, 14, ls.cable_assignment)
    print("TOTAL AFTER {}".format(new_cost))
    print('IMPROV: {}'.format(original_cost-new_cost))
    print(Counter(ls.degrees.values()))
    print()
    

TOTAL BEFORE 337267.9781319077
Total improvement: 0
TOTAL AFTER 337267.9781319077
IMPROV: 0.0
Counter({2: 583, 1: 25, 3: 11, 4: 6})

TOTAL BEFORE 345045.7082728962
Total improvement: 0
TOTAL AFTER 345045.7082728962
IMPROV: 0.0
Counter({2: 570, 1: 35, 4: 9, 3: 9, 5: 2})

TOTAL BEFORE 348642.97606753407
Total improvement: 0
TOTAL AFTER 348642.97606753407
IMPROV: 0.0
Counter({2: 564, 1: 36, 3: 17, 4: 7, 5: 1})

TOTAL BEFORE 339397.10488078697
Total improvement: 0
TOTAL AFTER 339397.10488078697
IMPROV: 0.0
Counter({2: 576, 1: 32, 3: 8, 4: 6, 5: 2, 6: 1})

TOTAL BEFORE 350518.14895278384
Total improvement: 0
TOTAL AFTER 350518.14895278384
IMPROV: 0.0
Counter({2: 564, 1: 38, 3: 14, 4: 7, 7: 1, 5: 1})

TOTAL BEFORE 354533.3599501997
Total improvement: 0
TOTAL AFTER 354533.3599501997
IMPROV: 0.0
Counter({2: 572, 1: 34, 3: 12, 4: 4, 5: 2, 8: 1})

TOTAL BEFORE 353468.222213322
Total improvement: 0
TOTAL AFTER 353468.222213322
IMPROV: 0.0
Counter({2: 568, 1: 35, 3: 17, 4: 3, 9: 1, 5: 1})



In [ ]:
plot_solution(coordinates, '/home/duc/mst_after_ls.png', ls.edges, 0, 0, 3)

In [19]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

def plot_solution(coordinates: np.ndarray,
                       output: str,
                       edges: List[List[Edge]],
                       value: float,
                       edge_cost: float,
                       partitions: int) -> None:
    plt.figure(figsize=(16,16), dpi=350)
    plt.axis('equal')

    coordinates_arr = np.array([tuple(x) for x in coordinates])
    for i,all_edges in enumerate(edges):
        edges_arr = np.array([tuple(list(x)) for x in all_edges])

        x = coordinates_arr[:,0].flatten()
        y = coordinates_arr[:,1].flatten()

        norm = matplotlib.colors.Normalize(vmin=0, vmax=len(edges), clip=True)
        mapper = cm.ScalarMappable(norm=norm, cmap=cm.cool)

        plt.plot(x[edges_arr.T], y[edges_arr.T], linestyle='-', color=mapper.to_rgba(i),
                 markerfacecolor='red', marker='o')
    plt.title("Cable cost/m: {}€ \nPartitions: {}\nTotal costs: {:0,.2f}€".format(edge_cost, partitions, value))
    plt.scatter(coordinates_arr[:,0], coordinates_arr[:,1], color='red', marker='o')
    plt.savefig(output, dpi=300)
    plt.close()
